# 03 模型对比：基准先跑，模型后上

不依赖 02 的内存，全部重新构建。比较 `naive_lag0` 与 Ridge / RF / GBDT。

In [2]:
# 自包含引导：从当前目录向上找到含 src/config.py 的仓库根，再挂到 sys.path
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / "src" / "config.py").exists() and p != p.parent:
    p = p.parent
sys.path.insert(0, str(p / "src"))
print("仓库根:", p)

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error

import data_io, features as F, split as S, models as M, evaluate as E

d = F.build_features(data_io.for_market(data_io.load_raw(), "DA"), h=1)
parts, spans = S.time_split(d)
tr, va, te = parts["train"], parts["valid"], parts["test"]

Xtr, ytr = tr[F.FEATURES], tr["y"]
Xva, yva = va[F.FEATURES], va["y"]
Xte, yte = te[F.FEATURES], te["y"]
print("train/valid/test =", len(tr), len(va), len(te))

仓库根: /Users/xian/Desktop/electricity-price
[features] h=1 丢弃 193 行（lag96 热身 + 末尾 1 行），保留 36959
train/valid/test = 22080 7296 7391


In [3]:
rows = []
rows.append({"model": "naive_lag0", **E.metrics(yte, te["lag0"])})
rows.append({"model": "naive_yday", **E.metrics(yte, te["lag96"])})

fitted, preds = {}, {}
for name in ["ridge", "rf", "gbdt"]:
    m = M.make_model(name)
    m.fit(Xtr, ytr)
    pred = m.predict(Xte)
    fitted[name], preds[name] = m, pred
    rows.append({"model": name, **E.metrics(yte, pred)})
    print(f"{name:6s} valid MAE={mean_absolute_error(yva, m.predict(Xva)):7.2f}")

cmp = pd.DataFrame(rows).sort_values("mae")
cmp

ridge  valid MAE=  32.10
rf     valid MAE=  33.43
gbdt   valid MAE=  33.84


,model,mae,rmse,wape
4,gbdt,22.413177,41.489463,0.053734
0,naive_lag0,22.448190,44.297416,0.053818
3,rf,22.505568,43.784118,0.053956
2,ridge,24.701689,43.726997,0.059221
1,naive_yday,59.559751,93.595257,0.142792


In [4]:
# 置换重要性：在验证集抽样 2000 行上打乱一列，看误差涨多少
# （全量 7296 行 × 400 棵树太慢，抽样不影响排名结论）
from sklearn.inspection import permutation_importance
import config as C

Xs = Xva.sample(2000, random_state=C.SEED)
ys = yva.loc[Xs.index]
r = permutation_importance(fitted["rf"], Xs, ys, n_repeats=5, random_state=C.SEED)
pd.Series(r.importances_mean, index=F.FEATURES).sort_values(ascending=False).round(3)

lag0            0.346
lag1            0.146
roll_mean_4     0.060
lag2            0.021
period_sin      0.016
lag4            0.008
period_id       0.007
lag96           0.004
roll_mean_96    0.003
period_cos      0.003
roll_std_96     0.002
lag96_diff      0.001
dow             0.000
is_weekend      0.000
lag192         -0.001
roll_max_96    -0.001
dtype: float64